In [46]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ============================================================
# 2. REPRODUCTIBILITÉ
# ============================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. DEVICE : CPU OU GPU
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device utilisé :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Aucun GPU CUDA détecté : utilisation du CPU")


Device utilisé : cpu
Aucun GPU CUDA détecté : utilisation du CPU


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "google-t5/t5-small"
)
#Détermine combien de tokens différents le modèle peut représenter et prédire
print("Vocabulaire :", tokenizer.vocab_size) #dictionnaire de tokens 
#Le token de padding sert à compléter les séquences pour qu'elles aient la même longueur dans un batch.
print("Token de padding :", tokenizer.pad_token)
#Il indique au modèle qu'une séquence est terminée.
print("Token de fin :", tokenizer.eos_token)

Vocabulaire : 32100
Token de padding : <pad>
Token de fin : </s>


In [116]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

train_texts = [
    text.strip()
    for text in dataset["train"]["text"]
    if text.strip()
]

val_texts = [
    text.strip()
    for text in dataset["validation"]["text"]
    if text.strip()
]

print("Nombre de textes d'entraînement :", len(train_texts))
print("Nombre de textes de validation :", len(val_texts))

print("\nExemple :")
print(train_texts[0])

c:\Users\aisso\Documents\GitHub\-Clustering-documents-BERT\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aisso\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 103157.96 exa

Nombre de textes d'entraînement : 23767
Nombre de textes de validation : 2461

Exemple :
= Valkyria Chronicles III =


In [123]:
corpus = train_texts[:5000]
val_corpus = val_texts[:500]

print("Textes d'entraînement :", len(corpus))
print("Textes de validation :", len(val_corpus))

Textes d'entraînement : 5000
Textes de validation : 500


In [124]:
# ============================================================
# 6. TOKENISATION DU CORPUS
# ============================================================

# On concatène les textes avec un token de fin entre eux.µ
text = tokenizer.eos_token.join(corpus)

tokens = tokenizer.encode(text)

print("Nombre total de tokens :", len(tokens))

print("\nPremier texte :")
print(corpus[0])

print("\nTokens correspondants :")
print(tokenizer.encode(corpus[0]))

print("\nDécodage :")
print(tokenizer.decode(tokenizer.encode(corpus[0])))

Token indices sequence length is longer than the specified maximum sequence length for this model (587275 > 512). Running this sequence through the model will result in indexing errors


Nombre total de tokens : 587275

Premier texte :
= Valkyria Chronicles III =

Tokens correspondants :
[3274, 3833, 3781, 52, 23, 9, 26143, 7, 6289, 3274, 1]

Décodage :
= Valkyria Chronicles III =</s>


In [135]:

import random
import torch
from torch.utils.data import Dataset

class DenoisingDataset(Dataset):

    def __init__(
        self,
        corpus,
        tokenizer,
        noise_density=0.25,
        seed=42,
        max_length=128
    ):
        self.examples = []
        rng = random.Random(seed)

        for sentence in corpus:
            token_ids = tokenizer.encode(
                sentence,
                add_special_tokens=False
            )

            # Réserver une place pour EOS
            token_ids = token_ids[:max_length - 1]

            if len(token_ids) < 3:
                continue

            # Nombre de tokens à masquer
            n_noise = max(
                1,
                min(
                    len(token_ids) - 1,
                    round(len(token_ids) * noise_density)
                )
            )

            # Choisir un span continu
            start = rng.randint(
                0,
                len(token_ids) - n_noise
            )
            end = start + n_noise

            sentinel_id = tokenizer.convert_tokens_to_ids(
                "<extra_id_0>"
            )

            corrupted_ids = (
                token_ids[:start]
                + [sentinel_id]
                + token_ids[end:]
                + [tokenizer.eos_token_id]
            )

            target_ids = (
                [sentinel_id]
                + token_ids[start:end]
                + [tokenizer.eos_token_id]
            )

            self.examples.append({
                "input_ids": torch.tensor(
                    corrupted_ids, dtype=torch.long
                ),
                "labels": torch.tensor(
                    target_ids, dtype=torch.long
                )
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


In [137]:
import torch

def collate_fn(batch):
    pad_id = tokenizer.pad_token_id

    # Trouver les longueurs maximales dans ce batch
    max_input_len = max(len(item["input_ids"]) for item in batch)
    max_label_len = max(len(item["labels"]) for item in batch)

    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for item in batch:
        input_ids = item["input_ids"]
        labels = item["labels"]

        # Padding des entrées
        input_padding = max_input_len - len(input_ids)
        padded_input = torch.cat([
            input_ids,
            torch.full((input_padding,), pad_id, dtype=torch.long)
        ])

        # 1 pour les vrais tokens, 0 pour le padding
        attention_mask = torch.cat([
            torch.ones(len(input_ids), dtype=torch.long),
            torch.zeros(input_padding, dtype=torch.long)
        ])

        # Padding des cibles avec -100 pour ignorer ces positions dans la loss
        label_padding = max_label_len - len(labels)
        padded_labels = torch.cat([
            labels,
            torch.full((label_padding,), -100, dtype=torch.long)
        ])

        input_ids_batch.append(padded_input)
        attention_mask_batch.append(attention_mask)
        labels_batch.append(padded_labels)

    return {
        "input_ids": torch.stack(input_ids_batch),
        "attention_mask": torch.stack(attention_mask_batch),
        "labels": torch.stack(labels_batch)
    }




In [141]:
BLOCK_SIZE = 128

train_dataset = DenoisingDataset(
    corpus,
    tokenizer,
    noise_density=0.25,
    seed=42,
    max_length=BLOCK_SIZE
)

val_dataset = DenoisingDataset(
    val_corpus,
    tokenizer,
    noise_density=0.25,
    seed=123,
    max_length=BLOCK_SIZE
)


from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

print("Batches d'entraînement :", len(train_loader))
print("Batches de validation :", len(val_loader))
print("Exemples train :", len(train_dataset))
print("Exemples validation :", len(val_dataset))

Batches d'entraînement : 1241
Batches de validation : 125
Exemples train : 4963
Exemples validation : 499


In [142]:

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 10. POSITION EMBEDDING
# ============================================================

class PositionalEmbedding(nn.Module):

    def __init__(self, block_size, embedding_dim):
        super().__init__()

        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

    def forward(self, x):
        batch_size, sequence_length = x.shape

        positions = torch.arange(
            sequence_length,
            device=x.device
        )

        return self.position_embedding(positions).unsqueeze(0)


# ============================================================
# 11. MODÈLE MINI-T5 SIMPLIFIÉ
# ============================================================

class MiniT5(nn.Module):

    def __init__(
        self,
        vocab_size,
        block_size,
        pad_token_id,
        embedding_dim=128,
        num_heads=4,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        self.block_size = block_size
        self.pad_token_id = pad_token_id

        # ----------------------------------------------------
        # Embedding partagé entre l'encodeur et le décodeur
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Embeddings positionnels
        self.encoder_position_embedding = PositionalEmbedding(
            block_size,
            embedding_dim
        )

        self.decoder_position_embedding = PositionalEmbedding(
            block_size,
            embedding_dim
        )

        # ----------------------------------------------------
        # ENCODEUR
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=4 * embedding_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embedding_dim)
        )

        # ----------------------------------------------------
        # DÉCODEUR
        # ----------------------------------------------------

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=4 * embedding_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embedding_dim)
        )

        # ----------------------------------------------------
        # PROJECTION VERS LE VOCABULAIRE
        # ----------------------------------------------------

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False
        )

        # Partage des poids entre embedding et projection
        self.lm_head.weight = self.token_embedding.weight

    # --------------------------------------------------------
    # DÉCALAGE DES LABELS POUR LE DÉCODEUR
    # --------------------------------------------------------

    def shift_right(self, labels):

        # Le décodeur commence avec le token PAD dans T5
        decoder_input_ids = labels.new_full(
            labels.shape,
            self.pad_token_id
        )

        # Chaque position reçoit le token cible précédent
        decoder_input_ids[:, 1:] = labels[:, :-1]

        # -100 sert uniquement à ignorer le padding dans la loss.
        # Il ne doit jamais être utilisé comme un identifiant de token.
        decoder_input_ids.masked_fill_(
            decoder_input_ids == -100,
            self.pad_token_id
        )

        return decoder_input_ids

    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    
    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,
        decoder_input_ids=None
    ):
        batch_size, input_length = input_ids.shape

        if input_length > self.block_size:
            raise ValueError("La séquence d'entrée dépasse block_size.")

        # A. ENCODEUR
        encoder_embeddings = (
            self.token_embedding(input_ids)
            + self.encoder_position_embedding(input_ids)
        )

        encoder_padding_mask = attention_mask == 0

        memory = self.encoder(
            encoder_embeddings,
            src_key_padding_mask=encoder_padding_mask
        )

        # B. PRÉPARATION DU DÉCODEUR
        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Fournir labels ou decoder_input_ids."
                )
            decoder_input_ids = self.shift_right(labels)

        decoder_length = decoder_input_ids.shape[1]

        if decoder_length > self.block_size:
            raise ValueError("La cible dépasse block_size.")

        decoder_embeddings = (
            self.token_embedding(decoder_input_ids)
            + self.decoder_position_embedding(decoder_input_ids)
        )

        # Masque causal : ne pas regarder les tokens futurs
        causal_mask = torch.triu(
            torch.ones(
                decoder_length,
                decoder_length,
                dtype=torch.bool,
                device=input_ids.device
            ),
            diagonal=1
        )

        # Masque du padding du décodeur
        if labels is not None:
            decoder_padding_mask = torch.zeros(
                decoder_input_ids.shape,
                dtype=torch.bool,
                device=input_ids.device
            )
            decoder_padding_mask[:, 1:] = labels[:, :-1] == -100
        else:
            # En génération, conserver le token de départ PAD
            # et les tokens déjà générés.
            decoder_padding_mask = torch.zeros(
                decoder_input_ids.shape,
                dtype=torch.bool,
                device=input_ids.device
            )

        # C. DÉCODEUR
        decoder_output = self.decoder(
            tgt=decoder_embeddings,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=decoder_padding_mask,
            memory_key_padding_mask=encoder_padding_mask
        )

        # D. LOGITS
        logits = self.lm_head(decoder_output)

        # Calculer la loss seulement pendant l'entraînement
        loss = None

        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )

        return {
            "logits": logits,
            "loss": loss
        }



In [130]:

model = MiniT5(
    vocab_size=len(tokenizer),
    block_size=128,
    pad_token_id=tokenizer.pad_token_id,
    embedding_dim=128,
    num_heads=4,
    num_layers=2,
    dropout=0.1
).to(device)

model = model.to(device)
print(model)


MiniT5(
  (token_embedding): Embedding(32100, 128)
  (encoder_position_embedding): PositionalEmbedding(
    (position_embedding): Embedding(128, 128)
  )
  (decoder_position_embedding): PositionalEmbedding(
    (position_embedding): Embedding(128, 128)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, e

C:\Users\aisso\AppData\Local\Temp\ipykernel_17368\861798138.py:86: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [147]:

from transformers import Adafactor

optimizer = Adafactor(
    model.parameters(),
    lr=1e-3,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
    clip_threshold=1.0,
    weight_decay=0.0
)

print("Optimiseur Adafactor créé avec succès.")


Optimiseur Adafactor créé avec succès.


In [149]:

# ============================================================
# 15. SCHEDULER DU LEARNING RATE
# ============================================================

from transformers import get_linear_schedule_with_warmup

EPOCHS = 30

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


In [ ]:
# ============================================================
# 16. PRÉENTRAÎNEMENT DE MINI-T5 : DENOISING
# ============================================================

EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()
    total_train_loss = 0.0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Réinitialiser les gradients
        optimizer.zero_grad()

        # Forward : encodeur + décodeur
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs["loss"]

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Mise à jour des paramètres
        optimizer.step()

        # Mise à jour du learning rate
        scheduler.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"| Train Loss = {average_train_loss:.4f} "
        f"| LR = {current_lr:.2e}"
    )

In [ ]:

# ============================================================
# 17. ÉVALUATION DE MINI-T5
# ============================================================

model.eval()
total_val_loss = 0.0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        total_val_loss += outputs["loss"].item()

average_val_loss = total_val_loss / len(val_loader)

print(f"Validation Loss = {average_val_loss:.4f}")



Validation Loss : 28.26179313659668


In [115]:
model.eval()

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        logits = outputs["logits"]
        predictions = logits.argmax(dim=-1)

        for j in range(input_ids.size(0)):
            print("\nExemple de validation :")

            print("Entrée :",
                  tokenizer.decode(
                      input_ids[j],
                      skip_special_tokens=False
                  ))

            print("Cible :",
                  tokenizer.decode(
                      labels[j][labels[j] != -100],
                      skip_special_tokens=False
                  ))

            print("Prédiction :",
                  tokenizer.decode(
                      predictions[j],
                      skip_special_tokens=False
                  ))

        if i == 1:
            break


Exemple de validation :
Entrée : A larger corp<extra_id_0> more linguistic information.</s><pad>
Cible : <extra_id_0>us generally provides</s>
Prédiction : <extra_id_0> modelus generally</s>

Exemple de validation :
Entrée : Deep learning<extra_id_0> useful representations from data.</s><pad>
Cible : <extra_id_0> models learn</s>
Prédiction : <extra_id_0> model models learn</s>

Exemple de validation :
Entrée : The model does not memorize language rules<extra_id_0></s><pad>
Cible : <extra_id_0> explicitly.</s>
Prédiction : <extra_id_0> token Lan</s></s>

Exemple de validation :
Entrée : A Transformer can process many tokens in parallel<extra_id_0></s>
Cible : <extra_id_0> during training.</s>
Prédiction : <extra_id_0> model during training</s>
